# G1 · Capa SILVER · Unificación agroambiental

**Grupo 1** · Ruta agroambiental · Pipeline E12 → producto E13 · Responsable: Brayan

**Pregunta del producto:** ¿qué cultivos tienen mejor rendimiento por municipio y año?

Este notebook implementa la capa SILVER del diagrama `Diagrama ejercicio 12.excalidraw`. Recibe la salida RAW de Rubén, une EVA, DIVIPOLA y AGROSAVIA **sin multiplicar filas**, publica un Parquet particionado para GOLD (Julián) y deja registro de la ejecución. El plan completo de la capa está en [`02_SILVER_Brayan.md`](../plan_grupo_1/02_SILVER_Brayan.md).

| Paso del diagrama | Sección |
|---|---|
| Verificación de fuentes ya almacenadas (antiduplicidad) | 5 y 12 |
| Unificación de datos | 6 a 10 |
| Conversión a Parquet y almacenamiento | 11 a 13 |
| Limpieza de memoria y caché | 15 |

**Entrada** (`ENTRADA_DIR`): `eva.parquet`, `divipola.parquet`, `agrosavia.parquet` y `controles_raw.json` según el contrato RAW.
**Salida** (`kit/salidas/grupo_1/<run_id>/silver/`): `dim_municipio.parquet`, `suelo_municipio_anio.parquet`, `unificado/` (Hive `anio=…/codigo_departamento=…`), `sin_correspondencia.parquet`, `controles_silver.json` y `entorno.json`.

**Cómo ejecutar.** Desde la raíz del repositorio genera el fixture una vez (`python soluciones/plan_grupo_1/fixtures/generar_fixture_raw.py`) y usa *Kernel → Restart & Run All*. Para datos reales cambia `ENTRADA_DIR` en la sección 1 por la carpeta `raw/` que entregue Rubén. Ejecuta siempre de arriba abajo con el kernel reiniciado: varias celdas dependen del estado de las anteriores.

**Historial de versiones**

| Versión | Fecha | Cambio |
|---|---|---|
| 1.0.0 | 2026-10-31 | Versión inicial: contrato RAW, alias territoriales, suelo municipio-año, unificado particionado, antiduplicidad, entorno y registro. |

## 1. Parámetros

Todo lo que cambia entre ejecuciones está en esta celda:

- `ENTRADA_DIR`: carpeta RAW. Por defecto el fixture; con datos reales, la carpeta `raw/` de Rubén.
- `SALIDA_BASE`: raíz de salidas del grupo (`kit/salidas/grupo_1`, fuera de Git).
- `RUN_ID` y `SALIDA_DIR`: con `None` se deciden en la sección 5 (se reutiliza el identificador de la carpeta RAW cuando corresponde).
- `FORZAR`: `True` reprocesa aunque exista una ejecución válida con la misma entrada.
- `RESPONSABLE`, `EQUIPO_DESCRIPCION`, `VERSION_NOTEBOOK`, `VERSION_REGLAS`: identifican quién, dónde y con qué versión del código y de las reglas se produjo la salida. Cambia `VERSION_REGLAS` cada vez que modifiques una regla (por ejemplo, la tabla de alias); eso obliga a reprocesar.

`encontrar_repositorio` es la misma función de `soluciones/E12_solucion.ipynb`, con `kit/fuentes.json` (manifiesto agroambiental) como marcador. `rel()` muestra las rutas relativas al repositorio para no registrar rutas personales.

In [1]:
from pathlib import Path
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version
from uuid import uuid4
from zoneinfo import ZoneInfo
import ctypes
import gc
import hashlib
import json
import os
import platform
import re
import subprocess
import sys
import unicodedata

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_repositorio(inicio=None):
    inicio = Path.cwd() if inicio is None else Path(inicio)
    for candidato in (inicio, *inicio.parents):
        if (candidato / 'kit' / 'fuentes.json').is_file():
            return candidato
        anidado = candidato / 'big-data-postgraduate'
        if (anidado / 'kit' / 'fuentes.json').is_file():
            return anidado
    raise FileNotFoundError('No se encontró la raíz de big-data-postgraduate')


REPOSITORIO = encontrar_repositorio()

# --- Parámetros de la ejecución ---
RUN_ID = None
ENTRADA_DIR = REPOSITORIO / 'soluciones' / 'plan_grupo_1' / 'fixtures' / 'raw'
# ENTRADA_DIR = REPOSITORIO / 'kit' / 'salidas' / 'grupo_1' / '<run_id>' / 'raw'   # salida real de Rubén
SALIDA_BASE = REPOSITORIO / 'kit' / 'salidas' / 'grupo_1'
SALIDA_DIR = None
FORZAR = False

# --- Versionamiento y equipo de ejecución ---
RESPONSABLE = 'Brayan'
EQUIPO_DESCRIPCION = 'Workstation personal · Fedora 44'
VERSION_NOTEBOOK = 'G1_02_silver 1.0.0'
VERSION_REGLAS = 'g1-silver-v1'
NOTEBOOK = 'soluciones/grupo_1/G1_02_silver.ipynb'
ZONA = ZoneInfo('America/Bogota')


def rel(ruta):
    ruta = Path(ruta)
    return ruta.relative_to(REPOSITORIO).as_posix() if ruta.is_relative_to(REPOSITORIO) else str(ruta)


print('Repositorio:', REPOSITORIO.name)
print('Entrada:', rel(ENTRADA_DIR))
print('Salidas:', rel(SALIDA_BASE))

Repositorio: big-data-postgraduate-bpineda
Entrada: soluciones/plan_grupo_1/fixtures/raw
Salidas: kit/salidas/grupo_1


## 2. Entorno de ejecución y versionamiento

Al unificar los cuatro notebooks del grupo hay que saber con qué Python, librerías, código y equipo se produjo cada capa. Esta celda arma el bloque `ENTORNO`, que se guarda en `silver/entorno.json`, dentro de `controles_silver.json` y, resumido, en el registro de ejecuciones.

- **Sistema:** plataforma, si corre en WSL, arquitectura, CPU, núcleos lógicos y RAM total. Se leen de `/proc` (Linux y WSL); en otros sistemas quedan en `None`. No se registra el nombre del equipo ni el usuario.
- **Python y librerías:** versión instalada de cada librería que usa la capa, comparada con la fijada en `kit/requirements.txt` y `kit/requirements_notebooks.txt`. Una diferencia es una **advertencia**: no invalida la ejecución, pero debe resolverse antes de unificar.
- **Git:** rama, commit y si este notebook tiene cambios sin commit. Así se sabe qué versión exacta del código produjo la salida.

In [2]:
def leer_proc(ruta, clave):
    """Valor de 'clave' en un archivo de /proc (Linux y WSL); None si no existe."""
    try:
        for linea in Path(ruta).read_text(encoding='utf-8').splitlines():
            if linea.startswith(clave):
                return linea.split(':', 1)[1].strip()
    except OSError:
        return None
    return None


def git(*argumentos):
    try:
        resultado = subprocess.run(['git', '-C', str(REPOSITORIO), *argumentos],
                                   capture_output=True, text=True, timeout=10, check=True)
        return resultado.stdout.strip()
    except (OSError, subprocess.SubprocessError):
        return None


requeridas = {}
for archivo in ['kit/requirements.txt', 'kit/requirements_notebooks.txt']:
    for linea in (REPOSITORIO / archivo).read_text(encoding='utf-8').splitlines():
        if '==' in linea and not linea.lstrip().startswith('#'):
            nombre, fijada = linea.split('==', 1)
            requeridas[nombre.strip().lower()] = fijada.strip()

librerias = {}
for nombre in ['pandas', 'numpy', 'pyarrow', 'duckdb', 'ipykernel']:
    try:
        instalada = version(nombre)
    except PackageNotFoundError:
        instalada = None
    requerida = requeridas.get(nombre)
    librerias[nombre] = {'instalada': instalada, 'requerida': requerida,
                         'coincide': requerida is None or instalada == requerida}

memoria_total = leer_proc('/proc/meminfo', 'MemTotal')
version_kernel = leer_proc('/proc/version', 'Linux') or ''
interprete = Path(sys.executable)
cambios_notebook = git('status', '--porcelain', '--', NOTEBOOK)

ENTORNO = {
    'capa': 'silver',
    'responsable': RESPONSABLE,
    'equipo_descripcion': EQUIPO_DESCRIPCION,
    'version_notebook': VERSION_NOTEBOOK,
    'version_reglas': VERSION_REGLAS,
    'fecha': datetime.now(ZONA).isoformat(timespec='seconds'),
    'zona_horaria': str(ZONA),
    'sistema': {
        'plataforma': platform.platform(),
        'wsl': 'microsoft' in (platform.release() + version_kernel).lower(),
        'arquitectura': platform.machine(),
        'cpu': leer_proc('/proc/cpuinfo', 'model name') or platform.processor() or None,
        'nucleos_logicos': os.cpu_count(),
        'ram_total_gb': round(int(memoria_total.split()[0]) / 1024**2, 1) if memoria_total else None,
    },
    'python': {
        'version': platform.python_version(),
        'implementacion': platform.python_implementation(),
        'interprete': rel(interprete) if interprete.is_relative_to(REPOSITORIO) else interprete.name,
    },
    'librerias': librerias,
    'git': {
        'rama': git('rev-parse', '--abbrev-ref', 'HEAD'),
        'commit': git('rev-parse', '--short', 'HEAD'),
        'notebook_con_cambios_sin_commit': None if cambios_notebook is None else bool(cambios_notebook),
    },
}

sistema = ENTORNO['sistema']
display(pd.DataFrame([
    ('Responsable', RESPONSABLE),
    ('Equipo', EQUIPO_DESCRIPCION),
    ('Versión del notebook', VERSION_NOTEBOOK),
    ('Versión de reglas', VERSION_REGLAS),
    ('Fecha', ENTORNO['fecha']),
    ('Sistema', sistema['plataforma']),
    ('WSL', sistema['wsl']),
    ('CPU', f"{sistema['cpu']} · {sistema['nucleos_logicos']} núcleos lógicos"),
    ('RAM total (GB)', sistema['ram_total_gb']),
    ('Python', f"{ENTORNO['python']['version']} ({ENTORNO['python']['interprete']})"),
    ('Git', f"{ENTORNO['git']['rama']} @ {ENTORNO['git']['commit']}"),
    ('Notebook con cambios sin commit', ENTORNO['git']['notebook_con_cambios_sin_commit']),
], columns=['campo', 'valor']).set_index('campo'))
display(pd.DataFrame.from_dict(librerias, orient='index').rename_axis('libreria'))

distintas = [nombre for nombre, datos in librerias.items() if not datos['coincide']]
if distintas:
    print('ADVERTENCIA: versiones distintas a las fijadas en kit/requirements*.txt:', ', '.join(distintas))
else:
    print('Todas las librerías coinciden con las versiones fijadas del curso.')

,valor
campo,
Responsable,Brayan
Equipo,Workstation personal · Fedora 44
Versión del notebook,G1_02_silver 1.0.0
Versión de reglas,g1-silver-v1
Fecha,2026-10-31T12:07:19-05:00
Sistema,Linux-7.2.13-200.fc44.x86_64-x86_64-with-glibc...
WSL,False
CPU,AMD Ryzen 7 9800X3D 8-Core Processor · 8 núcle...
RAM total (GB),30.5


,instalada,requerida,coincide
libreria,,,
pandas,2.2.3,2.2.3,True
numpy,2.2.6,2.2.6,True
pyarrow,19.0.1,19.0.1,True
duckdb,1.4.1,1.4.1,True
ipykernel,6.30.1,6.30.1,True


Todas las librerías coinciden con las versiones fijadas del curso.


## 3. Funciones de control

- `firma_dataframe` y `validar_contrato` se copian de `soluciones/E12_solucion.ipynb` (sección 2). La firma ordena las filas, convierte todo a texto y calcula SHA-256: dos tablas con el mismo contenido tienen la misma firma aunque los archivos Parquet no sean idénticos byte a byte.
- `firma_conjunta` combina varias firmas en una sola (para la entrada y para la salida completa de la capa).
- `norm` es la misma normalización de nombres de `kit/talleres.py`: sin tildes, mayúsculas, solo letras, números y espacios simples. Debe coincidir con la que usó RAW para construir `clave_territorial`.
- `memoria_mb` lee `RssAnon` (memoria de trabajo del proceso). No se usa RSS porque los Parquet se leen con *mmap* y RSS contaría páginas del archivo.
- `comprobar` registra cada control en `CONTROLES`, lo imprime y detiene el notebook si falla. Si algo falla, la salida nunca queda marcada como válida.

In [3]:
def firma_dataframe(datos, columnas, orden):
    normalizado = datos.loc[:, columnas].copy()
    normalizado = normalizado.sort_values(orden, kind='stable').reset_index(drop=True)
    normalizado = normalizado.astype('string').fillna('<NA>')
    texto = normalizado.to_csv(index=False, lineterminator='\n')
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()


def firma_conjunta(firmas):
    texto = json.dumps(firmas, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()


def validar_contrato(datos, columnas_esperadas, nombre):
    actuales = datos.columns.tolist()
    if actuales != columnas_esperadas:
        faltantes = sorted(set(columnas_esperadas) - set(actuales))
        extras = sorted(set(actuales) - set(columnas_esperadas))
        raise ValueError(
            f'Contrato incumplido en {nombre}; faltantes={faltantes}; extras={extras}; '
            'también debe conservarse el orden de columnas'
        )


def norm(x):
    x = ''.join(c for c in unicodedata.normalize('NFKD', str(x)) if not unicodedata.combining(c))
    return ' '.join(re.sub(r'[^A-Z0-9 ]', ' ', x.upper()).split())


def memoria_mb():
    valor = leer_proc('/proc/self/status', 'RssAnon')
    return round(int(valor.split()[0]) / 1024, 1) if valor else None


CONTROLES = {}


def comprobar(nombre, condicion, detalle=''):
    CONTROLES[nombre] = bool(condicion)
    marca = 'OK   ' if condicion else 'FALLA'
    print(f'{marca} {nombre}' + (f' · {detalle}' if detalle else ''))
    assert condicion, f'Control fallido: {nombre} {detalle}'

## 4. Contrato de entrada (RAW)

SILVER no empieza si RAW no terminó bien. Se comprueba:

1. `controles_raw.json` tiene `estado = "valida"`.
2. Cada Parquet tiene **exactamente** las columnas del contrato de `01_RAW_Ruben.md`, en el mismo orden.
3. Los códigos DANE son texto de 2 y 5 dígitos, y los dos primeros dígitos del municipio coinciden con el departamento en EVA.
4. DIVIPOLA es única por código y por `clave_territorial`.
5. `clave_territorial` está normalizada igual que con `norm()` en DIVIPOLA y AGROSAVIA. Si RAW normalizara distinto, el enlace por nombre fallaría en silencio.
6. `fila_fuente` es única en EVA y AGROSAVIA: es la clave de trazabilidad hacia los CSV originales.

In [4]:
CONTRATO_RAW = {
    'eva': ['codigo_departamento', 'codigo_municipio', 'departamento', 'municipio', 'grupo_cultivo', 'cultivo',
            'anio', 'periodo', 'area_sembrada_ha', 'area_cosechada_ha', 'produccion_t',
            'rendimiento_publicado_t_ha', 'ciclo', 'estado_fisico', 'codigo_cultivo', 'fila_fuente'],
    'divipola': ['codigo_departamento', 'codigo_municipio', 'departamento', 'municipio', 'tipo',
                 'longitud', 'latitud', 'clave_territorial'],
    'agrosavia': ['id_muestra', 'fila_fuente', 'fecha', 'anio', 'departamento', 'municipio', 'clave_territorial',
                  'cultivo', 'ph', 'materia_organica', 'fosforo', 'aluminio', 'calcio', 'magnesio', 'potasio',
                  'cic', 'conductividad'],
}

controles_raw = json.loads((ENTRADA_DIR / 'controles_raw.json').read_text(encoding='utf-8'))
comprobar('raw_valida', controles_raw.get('estado') == 'valida', f"estado RAW = {controles_raw.get('estado')}")

eva = pd.read_parquet(ENTRADA_DIR / 'eva.parquet')
divipola = pd.read_parquet(ENTRADA_DIR / 'divipola.parquet')
agrosavia = pd.read_parquet(ENTRADA_DIR / 'agrosavia.parquet')
for nombre, tabla in [('eva', eva), ('divipola', divipola), ('agrosavia', agrosavia)]:
    validar_contrato(tabla, CONTRATO_RAW[nombre], nombre)
print('Columnas de entrada conforme al contrato RAW.')


def todos_cumplen(serie, patron):
    return bool(serie.astype('string').str.fullmatch(patron).fillna(False).all())


for nombre, tabla in [('eva', eva), ('divipola', divipola)]:
    comprobar(f'{nombre}_codigo_departamento_2_digitos', todos_cumplen(tabla['codigo_departamento'], r'\d{2}'))
    comprobar(f'{nombre}_codigo_municipio_5_digitos', todos_cumplen(tabla['codigo_municipio'], r'\d{5}'))
comprobar('eva_municipio_en_su_departamento', eva['codigo_municipio'].str[:2].eq(eva['codigo_departamento']).all())
comprobar('divipola_codigo_unico', divipola['codigo_municipio'].is_unique)
comprobar('divipola_clave_unica', divipola['clave_territorial'].is_unique)
for nombre, tabla in [('divipola', divipola), ('agrosavia', agrosavia)]:
    esperada = tabla['departamento'].map(norm) + '|' + tabla['municipio'].map(norm)
    comprobar(f'{nombre}_clave_normalizada', tabla['clave_territorial'].eq(esperada).all())
comprobar('eva_fila_fuente_unica', eva['fila_fuente'].is_unique)
comprobar('agrosavia_fila_fuente_unica', agrosavia['fila_fuente'].is_unique)

display(pd.DataFrame({'filas': {'eva': len(eva), 'divipola': len(divipola), 'agrosavia': len(agrosavia)}}))

OK    raw_valida · estado RAW = valida
Columnas de entrada conforme al contrato RAW.
OK    eva_codigo_departamento_2_digitos
OK    eva_codigo_municipio_5_digitos
OK    divipola_codigo_departamento_2_digitos
OK    divipola_codigo_municipio_5_digitos
OK    eva_municipio_en_su_departamento
OK    divipola_codigo_unico
OK    divipola_clave_unica
OK    divipola_clave_normalizada
OK    agrosavia_clave_normalizada
OK    eva_fila_fuente_unica
OK    agrosavia_fila_fuente_unica


,filas
eva,8
divipola,4
agrosavia,7


### 4.1 Entorno de RAW frente a SILVER

Si RAW registró su bloque `entorno` (convención de versionamiento del grupo), se comparan Python y librerías. Una diferencia no detiene la ejecución, pero es lo primero que se revisa si al unificar los notebooks aparecen resultados distintos.

In [5]:
entorno_raw = controles_raw.get('entorno')
if not entorno_raw:
    print('RAW no registró el bloque "entorno": pedirlo en la revisión del PR (convención de versionamiento).')
else:
    filas = [('python', entorno_raw.get('python', {}).get('version'), ENTORNO['python']['version'])]
    for nombre, datos in ENTORNO['librerias'].items():
        filas.append((nombre, entorno_raw.get('librerias', {}).get(nombre, {}).get('instalada'), datos['instalada']))
    comparacion = pd.DataFrame(filas, columns=['componente', 'raw', 'silver'])
    comparacion['coincide'] = [None if raw is None else raw == silver
                               for raw, silver in zip(comparacion['raw'], comparacion['silver'])]
    print('RAW:', entorno_raw.get('responsable'), '·', entorno_raw.get('version_notebook'))
    display(comparacion.set_index('componente'))
    if comparacion['coincide'].eq(False).any():
        print('ADVERTENCIA: RAW y SILVER se ejecutaron con versiones distintas.')

RAW: fixture · generar_fixture_raw.py


,raw,silver,coincide
componente,,,
python,3.12.14,3.12.14,True
pandas,2.2.3,2.2.3,True
numpy,2.2.6,2.2.6,True
pyarrow,19.0.1,19.0.1,True
duckdb,None,1.4.1,None
ipykernel,None,6.30.1,None


## 5. Verificación de entradas ya procesadas (antiduplicidad)

Paso transversal del diagrama: *verificación de fuentes de datos ya almacenadas para evitar duplicidades*.

- `firma_entrada` combina la firma de **contenido** de las tres tablas de entrada y `VERSION_REGLAS`. Se usa el contenido y no el hash del archivo porque dos ejecuciones de RAW con los mismos datos producen Parquet que no son idénticos byte a byte (E12 §7).
- Si `registro_silver.jsonl` ya tiene una ejecución `valida` con la misma firma y su carpeta existe, no se vuelve a escribir: la sección 12 comprueba que el resultado recalculado es igual al publicado y las secciones siguientes verifican la carpeta existente.
- Si no, se decide `RUN_ID`: se reutiliza el de la carpeta RAW cuando la entrada es `kit/salidas/grupo_1/<run_id>/raw` y esa ejecución aún no tiene `silver/`; en otro caso se genera uno nuevo. Nunca se escribe sobre una carpeta existente.

In [6]:
FIRMAS_ENTRADA = {
    'eva': firma_dataframe(eva, CONTRATO_RAW['eva'], ['fila_fuente']),
    'divipola': firma_dataframe(divipola, CONTRATO_RAW['divipola'], ['codigo_municipio']),
    'agrosavia': firma_dataframe(agrosavia, CONTRATO_RAW['agrosavia'], ['fila_fuente']),
    'version_reglas': VERSION_REGLAS,
}
FIRMA_ENTRADA = firma_conjunta(FIRMAS_ENTRADA)

REGISTRO = SALIDA_BASE / 'registro_silver.jsonl'
previas = []
if REGISTRO.is_file():
    previas = [json.loads(linea) for linea in REGISTRO.read_text(encoding='utf-8').splitlines() if linea.strip()]
PREVIA = next((registro for registro in reversed(previas)
               if registro['firma_entrada'] == FIRMA_ENTRADA and registro['estado'] == 'valida'
               and (REPOSITORIO / registro['ruta_salida']).is_dir()), None)
YA_PROCESADO = PREVIA is not None and not FORZAR

if YA_PROCESADO:
    RUN_ID = PREVIA['run_id']
    SALIDA_DIR = REPOSITORIO / PREVIA['ruta_salida']
    print(f'Ya procesado: la misma entrada y reglas se publicaron en {rel(SALIDA_DIR)}. No se escribirá de nuevo.')
else:
    if RUN_ID is None:
        en_estructura = ENTRADA_DIR.name == 'raw' and ENTRADA_DIR.parent.parent == SALIDA_BASE
        libre = en_estructura and not (ENTRADA_DIR.parent / 'silver').exists()
        RUN_ID = ENTRADA_DIR.parent.name if libre else uuid4().hex[:12]
    SALIDA_DIR = SALIDA_DIR or SALIDA_BASE / RUN_ID / 'silver'
    comprobar('salida_nueva', not SALIDA_DIR.exists(), rel(SALIDA_DIR))
    motivo = 'FORZAR = True' if PREVIA else 'entrada no procesada antes'
    print(f'Nueva ejecución {RUN_ID} ({motivo}) → {rel(SALIDA_DIR)}')
print('firma_entrada:', FIRMA_ENTRADA[:16], '…', '· ejecuciones previas en el registro:', len(previas))

Ya procesado: la misma entrada y reglas se publicaron en kit/salidas/grupo_1/c9512e696006/silver. No se escribirá de nuevo.
firma_entrada: 39623be9824f258c … · ejecuciones previas en el registro: 1


## 6. Dimensión de municipios

`dim_municipio` es DIVIPOLA con las columnas que necesitan GOLD y Presentación (códigos, nombres oficiales y coordenadas). Su clave es `codigo_municipio`; se comprueba que sea única porque cualquier unión con ella debe ser *muchos a uno*.

In [7]:
COLUMNAS_DIM = ['codigo_departamento', 'codigo_municipio', 'departamento', 'municipio', 'longitud', 'latitud']
dim_municipio = divipola[COLUMNAS_DIM].sort_values('codigo_municipio').reset_index(drop=True)
comprobar('dim_clave_unica', dim_municipio['codigo_municipio'].is_unique, f'{len(dim_municipio):,} municipios')
display(dim_municipio.head())

OK    dim_clave_unica · 4 municipios


,codigo_departamento,codigo_municipio,departamento,municipio,longitud,latitud
0,05,05001,ANTIOQUIA,MEDELLÍN,-75.581775,6.246631
1,15,15001,BOYACÁ,TUNJA,-73.355539,5.539880
2,15,15248,BOYACÁ,EL ESPINO,-72.497007,6.483027
3,54,54001,NORTE DE SANTANDER,SAN JOSÉ DE CÚCUTA,-72.508178,7.905725


## 7. Enlace AGROSAVIA ↔ DIVIPOLA

AGROSAVIA no trae código DANE: solo departamento y municipio escritos a mano. El enlace se hace por `clave_territorial` (`DEPARTAMENTO|MUNICIPIO` normalizado).

Con los datos completos, el 97,7 % de las muestras enlaza directamente. Casi todo el resto son **nombres oficiales que cambiaron** (Cúcuta → San José de Cúcuta, Cali → Santiago de Cali, Mompós → Santa Cruz de Mompox…). Para recuperarlos se usa `ALIAS_TERRITORIAL`, una **regla declarada y versionada** (`VERSION_REGLAS`) con dos condiciones:

- solo renombres inequívocos **dentro del mismo departamento**;
- no se incluyen valores basura (`…|BORRAR`) ni casos con el departamento dudoso (por ejemplo `CUNDINAMARCA|TOCA`, que existe en Boyacá). Esos quedan en `sin_correspondencia`.

Se comprueba que cada alias que aparece en AGROSAVIA apunte a una clave que existe en DIVIPOLA y que ninguna clave de origen exista ya en DIVIPOLA (si existiera, el alias cambiaría un enlace correcto). La unión es `many_to_one`: pandas detiene la ejecución si una clave de DIVIPOLA estuviera repetida.

In [8]:
ALIAS_TERRITORIAL = {
    'NORTE DE SANTANDER|CUCUTA': 'NORTE DE SANTANDER|SAN JOSE DE CUCUTA',
    'VALLE DEL CAUCA|CALI': 'VALLE DEL CAUCA|SANTIAGO DE CALI',
    'CAUCA|SOTARA': 'CAUCA|SOTARA PAISPAMBA',
    'META|FUENTEDEORO': 'META|FUENTE DE ORO',
    'META|SAN LUIS DE CUBARRAL': 'META|CUBARRAL',
    'CAUCA|PIENDAMO': 'CAUCA|PIENDAMO TUNIA',
    'NARINO|CUASPUD': 'NARINO|CUASPUD CARLOSAMA',
    'TOLIMA|ARMERO GUAYABAL': 'TOLIMA|ARMERO',
    'SUCRE|TOLU VIEJO': 'SUCRE|SAN JOSE DE TOLUVIEJO',
    'BOYACA|ESPINO': 'BOYACA|EL ESPINO',
    'BOLIVAR|MOMPOS': 'BOLIVAR|SANTA CRUZ DE MOMPOX',
    'BOYACA|GUICAN': 'BOYACA|GUICAN DE LA SIERRA',
    'GUAINIA|BARRANCO MINAS': 'GUAINIA|BARRANCOMINAS',
    'SUCRE|SAN BENITO': 'SUCRE|SAN BENITO ABAD',
    'SUCRE|BETULIA': 'SUCRE|SAN JUAN DE BETULIA',
    'NARINO|PENOL': 'NARINO|EL PENOL',
    'CHOCO|BELEN DE BAJIRA': 'CHOCO|NUEVO BELEN DE BAJIRA',
    'CESAR|MANAURE': 'CESAR|MANAURE BALCON DEL CESAR',
}
claves_divipola = set(divipola['clave_territorial'])
alias_usados = {origen: destino for origen, destino in ALIAS_TERRITORIAL.items()
                if origen in set(agrosavia['clave_territorial'])}
comprobar('alias_usados_existen_en_divipola', set(alias_usados.values()) <= claves_divipola,
          f'{len(alias_usados)} de {len(ALIAS_TERRITORIAL)} alias aparecen en AGROSAVIA')
comprobar('alias_origen_no_existe_en_divipola', not set(ALIAS_TERRITORIAL) & claves_divipola)

agro = agrosavia.copy()
agro['clave_enlace'] = agro['clave_territorial'].map(ALIAS_TERRITORIAL).fillna(agro['clave_territorial'])
agro['por_alias'] = agro['clave_enlace'].ne(agro['clave_territorial'])
agro = agro.merge(
    divipola[['clave_territorial', 'codigo_municipio']].rename(columns={'clave_territorial': 'clave_enlace'}),
    on='clave_enlace', how='left', validate='many_to_one')
agro['enlazada'] = agro['codigo_municipio'].notna()

CONTEOS = {
    'agrosavia_entrada': len(agrosavia),
    'agrosavia_enlazadas': int(agro['enlazada'].sum()),
    'agrosavia_sin_enlace': int((~agro['enlazada']).sum()),
    'agrosavia_por_alias': int((agro['por_alias'] & agro['enlazada']).sum()),
}
comprobar('agrosavia_conserva_filas', len(agro) == len(agrosavia))
comprobar('agrosavia_enlazadas_mas_sin_enlace',
          CONTEOS['agrosavia_enlazadas'] + CONTEOS['agrosavia_sin_enlace'] == len(agrosavia))
print(f"Enlazadas: {CONTEOS['agrosavia_enlazadas']:,} de {len(agrosavia):,} "
      f"({100 * CONTEOS['agrosavia_enlazadas'] / len(agrosavia):.2f} %), "
      f"de ellas {CONTEOS['agrosavia_por_alias']:,} por alias.")
display(agro.loc[~agro['enlazada'], 'clave_territorial'].value_counts().head(10).rename('filas').to_frame())

OK    alias_usados_existen_en_divipola · 2 de 18 alias aparecen en AGROSAVIA
OK    alias_origen_no_existe_en_divipola
OK    agrosavia_conserva_filas
OK    agrosavia_enlazadas_mas_sin_enlace
Enlazadas: 6 de 7 (85.71 %), de ellas 2 por alias.


,filas
clave_territorial,
CAUCA|BORRAR,1


## 8. Suelo por municipio y año (DuckDB)

Cada municipio tiene muchas muestras de suelo y muchos registros de EVA. Unir muestra por muestra con cada registro de cultivo **multiplicaría filas** y duplicaría la producción. Por eso primero se agrega AGROSAVIA a una fila por `(codigo_municipio, anio)` y después se une.

Se usa DuckDB sobre el DataFrame de pandas: `count(*)` cuenta muestras y `median()` ignora los nulos (los `ND` de laboratorio que RAW convirtió en nulo). Controles: la clave es única y la suma de `muestras` es igual al número de muestras enlazadas.

In [9]:
con = duckdb.connect()
agro_enlazada = agro.loc[agro['enlazada'], ['codigo_municipio', 'anio', 'ph', 'materia_organica', 'fosforo']]
con.register('agro_enlazada', agro_enlazada)
suelo_municipio_anio = con.execute("""
    SELECT codigo_municipio,
           anio,
           count(*)::BIGINT         AS muestras,
           median(ph)               AS mediana_ph,
           median(materia_organica) AS mediana_materia_organica,
           median(fosforo)          AS mediana_fosforo
    FROM agro_enlazada
    GROUP BY codigo_municipio, anio
    ORDER BY codigo_municipio, anio
""").df()

comprobar('suelo_clave_unica', not suelo_municipio_anio.duplicated(['codigo_municipio', 'anio']).any(),
          f'{len(suelo_municipio_anio):,} municipio-año')
comprobar('suelo_muestras_conciliadas', suelo_municipio_anio['muestras'].sum() == CONTEOS['agrosavia_enlazadas'])
display(suelo_municipio_anio.head())

OK    suelo_clave_unica · 5 municipio-año
OK    suelo_muestras_conciliadas


,codigo_municipio,anio,muestras,mediana_ph,mediana_materia_organica,mediana_fosforo
0,15001,2019,1,5.2,4.5,18.0
1,15001,2021,2,5.5,5.0,20.0
2,15001,2022,1,NaN,5.0,20.0
3,15248,2022,1,4.5,3.0,8.0
4,54001,2021,1,7.0,2.0,15.0


## 9. Unificación: EVA como tabla base

El grano de la salida es **una fila por registro aceptado de EVA**, porque GOLD calcula rendimientos desde producción y área. Pasos:

1. EVA ⟕ `dim_municipio` por `codigo_municipio` (`many_to_one`). `en_divipola` indica si el código existe; los nombres oficiales vienen de DIVIPOLA y, si no hay correspondencia, se conservan los de EVA.
2. ⟕ `suelo_municipio_anio` por `(codigo_municipio, anio)` (`many_to_one`). Sin muestras, `muestras_suelo = 0` y `mediana_ph` nulo.
3. Columnas derivadas: `area_cosechada_m2 = area_cosechada_ha × 10 000` (granularidad en m² del diagrama) y `apto_rendimiento` (área cosechada > 0 y producción ≥ 0). Las filas no aptas **no se eliminan**: GOLD las filtra y la conciliación las sigue viendo.

Controles: mismas filas que EVA, y la producción y el área por año y total no cambian con las uniones.

In [10]:
COLUMNAS_UNIFICADO = ['codigo_departamento', 'departamento', 'codigo_municipio', 'municipio', 'anio', 'periodo',
                      'grupo_cultivo', 'cultivo', 'estado_fisico', 'ciclo', 'area_sembrada_ha', 'area_cosechada_ha',
                      'area_cosechada_m2', 'produccion_t', 'rendimiento_publicado_t_ha', 'apto_rendimiento',
                      'en_divipola', 'muestras_suelo', 'mediana_ph', 'fila_fuente']

eva_dim = eva.merge(
    dim_municipio[['codigo_municipio', 'departamento', 'municipio']].rename(
        columns={'departamento': 'departamento_divipola', 'municipio': 'municipio_divipola'}),
    on='codigo_municipio', how='left', validate='many_to_one', indicator=True)
eva_dim['en_divipola'] = eva_dim['_merge'].eq('both')
eva_dim['departamento'] = eva_dim['departamento_divipola'].fillna(eva_dim['departamento'])
eva_dim['municipio'] = eva_dim['municipio_divipola'].fillna(eva_dim['municipio'])

unificado = eva_dim.merge(
    suelo_municipio_anio[['codigo_municipio', 'anio', 'muestras', 'mediana_ph']].rename(
        columns={'muestras': 'muestras_suelo'}),
    on=['codigo_municipio', 'anio'], how='left', validate='many_to_one')
unificado['muestras_suelo'] = unificado['muestras_suelo'].fillna(0).astype('int64')
unificado['area_cosechada_m2'] = unificado['area_cosechada_ha'] * 10_000
unificado['apto_rendimiento'] = unificado['area_cosechada_ha'].gt(0) & unificado['produccion_t'].ge(0)
unificado = unificado[COLUMNAS_UNIFICADO]

comprobar('unificado_conserva_filas_eva', len(unificado) == len(eva), f'{len(unificado):,} filas')
medidas = ['produccion_t', 'area_cosechada_ha']
antes = eva.groupby('anio')[medidas].sum()
despues = unificado.groupby('anio')[medidas].sum()
comprobar('sumas_por_anio_sin_cambio',
          antes.index.equals(despues.index) and np.allclose(antes, despues, rtol=0, atol=1e-6))
comprobar('sumas_totales_sin_cambio', np.allclose(eva[medidas].sum(), unificado[medidas].sum(), rtol=0, atol=1e-6))

CONTEOS.update({
    'filas_eva_entrada': len(eva),
    'filas_unificado': len(unificado),
    'filas_aptas_rendimiento': int(unificado['apto_rendimiento'].sum()),
    'eva_sin_divipola': int((~unificado['en_divipola']).sum()),
    'cobertura_suelo_pct': round(float(unificado['muestras_suelo'].gt(0).mean() * 100), 2),
    'produccion_total_t': float(unificado['produccion_t'].sum()),
    'area_cosechada_total_ha': float(unificado['area_cosechada_ha'].sum()),
})
display(pd.Series(CONTEOS, name='valor', dtype=object).to_frame())
display(unificado.head())

OK    unificado_conserva_filas_eva · 8 filas
OK    sumas_por_anio_sin_cambio
OK    sumas_totales_sin_cambio


,valor
agrosavia_entrada,7
agrosavia_enlazadas,6
agrosavia_sin_enlace,1
agrosavia_por_alias,2
filas_eva_entrada,8
filas_unificado,8
filas_aptas_rendimiento,7
eva_sin_divipola,1
cobertura_suelo_pct,75.0
produccion_total_t,1809.0


,codigo_departamento,departamento,codigo_municipio,municipio,anio,periodo,grupo_cultivo,cultivo,estado_fisico,ciclo,area_sembrada_ha,area_cosechada_ha,area_cosechada_m2,produccion_t,rendimiento_publicado_t_ha,apto_rendimiento,en_divipola,muestras_suelo,mediana_ph,fila_fuente
0,15,BOYACÁ,15001,TUNJA,2021,2021A,Tubérculos y plátanos,Papa,En fresco,Transitorio,12.0,10.0,100000.0,200.0,20.0,True,True,2,5.5,2
1,15,BOYACÁ,15001,TUNJA,2021,2021B,Tubérculos y plátanos,Papa,En fresco,Transitorio,30.0,30.0,300000.0,900.0,30.0,True,True,2,5.5,3
2,15,BOYACÁ,15001,TUNJA,2021,2021A,Leguminosas,Arveja,En fresco,Transitorio,6.0,5.0,50000.0,25.0,5.0,True,True,2,5.5,4
3,15,BOYACÁ,15001,TUNJA,2022,2022A,Tubérculos y plátanos,Papa,En fresco,Transitorio,5.0,0.0,0.0,0.0,NaN,False,True,1,NaN,5
4,54,NORTE DE SANTANDER,54001,SAN JOSÉ DE CÚCUTA,2021,2021A,Cereales,Arroz,Paddy o cascara verde,Transitorio,100.0,100.0,1000000.0,500.0,5.0,True,True,1,7.0,7


## 10. Registros sin correspondencia

E12 exige conservar explícitamente lo que no enlaza. `sin_correspondencia` tiene una fila por clave no enlazada:

- `agrosavia`: claves territoriales sin municipio en DIVIPOLA (ni directo ni por alias);
- `eva`: códigos de municipio que no existen en DIVIPOLA.

Control: la suma de `filas` por fuente coincide con los conteos de las secciones 7 y 9.

In [11]:
sin_agro = (agro.loc[~agro['enlazada']].groupby('clave_territorial').size().rename('filas').reset_index()
            .rename(columns={'clave_territorial': 'clave'}).assign(fuente='agrosavia'))
sin_eva = (unificado.loc[~unificado['en_divipola']].groupby('codigo_municipio').size().rename('filas').reset_index()
           .rename(columns={'codigo_municipio': 'clave'}).assign(fuente='eva'))
sin_correspondencia = (pd.concat([sin_agro, sin_eva], ignore_index=True)[['fuente', 'clave', 'filas']]
                       .astype({'filas': 'int64'})
                       .sort_values(['fuente', 'filas', 'clave'], ascending=[True, False, True])
                       .reset_index(drop=True))

comprobar('sin_correspondencia_agrosavia', sin_agro['filas'].sum() == CONTEOS['agrosavia_sin_enlace'])
comprobar('sin_correspondencia_eva', sin_eva['filas'].sum() == CONTEOS['eva_sin_divipola'])
display(sin_correspondencia.head(10))

OK    sin_correspondencia_agrosavia
OK    sin_correspondencia_eva


,fuente,clave,filas
0,agrosavia,CAUCA|BORRAR,1
1,eva,25999,1


## 11. Firmas de contenido

Se calcula la firma de cada tabla de salida, ordenada por su clave, y una `firma_contenido` conjunta. Sirven para tres cosas: comprobar que lo releído es igual a lo escrito (sección 13), detectar si una reejecución con la misma entrada produce otro resultado (sección 12) y comparar las ejecuciones A y B en el notebook integrado.

In [12]:
TABLAS = {
    'unificado': unificado,
    'dim_municipio': dim_municipio,
    'suelo_municipio_anio': suelo_municipio_anio,
    'sin_correspondencia': sin_correspondencia,
}
ORDEN = {
    'unificado': ['fila_fuente'],
    'dim_municipio': ['codigo_municipio'],
    'suelo_municipio_anio': ['codigo_municipio', 'anio'],
    'sin_correspondencia': ['fuente', 'clave'],
}
FIRMAS = {nombre: firma_dataframe(tabla, list(tabla.columns), ORDEN[nombre]) for nombre, tabla in TABLAS.items()}
FIRMA_CONTENIDO = firma_conjunta(FIRMAS)
display(pd.Series({nombre: firma[:16] + '…' for nombre, firma in FIRMAS.items()}, name='firma').to_frame())
print('firma_contenido:', FIRMA_CONTENIDO[:16], '…')

,firma
unificado,5895951fecc19555…
dim_municipio,247861d6133aa426…
suelo_municipio_anio,7fef8eab8737be4e…
sin_correspondencia,0335441ab6b0ebe3…


firma_contenido: d4b72ec4748525dc …


## 12. Conversión a Parquet y almacenamiento

Única celda que escribe archivos.

- **Ya procesado:** no se escribe nada. Se comprueba que la firma recalculada es igual a la registrada: si no lo fuera, el proceso no es reproducible y hay que investigarlo.
- **Ejecución nueva:** se crea la carpeta (falla si ya existe) y se escriben las tablas pequeñas con pandas y `unificado/` con `COPY … PARTITION_BY` de DuckDB.

**Decisión de particionamiento.** El diagrama propone año y municipio. Con ~1.100 municipios por cada año y pocas filas por combinación, eso genera miles de archivos diminutos (más metadatos que datos). Se particiona por `anio` y `codigo_departamento` (~33 departamentos por año); `codigo_municipio` queda como columna y DuckDB filtra por él con las estadísticas de Parquet. DuckDB no guarda las columnas de partición dentro de los archivos: se reconstruyen desde las carpetas al leer.

In [13]:
def literal_sql(ruta):
    return "'" + Path(ruta).as_posix().replace("'", "''") + "'"


if YA_PROCESADO:
    comprobar('firma_igual_a_la_publicada', FIRMA_CONTENIDO == PREVIA['firma_contenido'])
    print('No se escribe: se reutiliza', rel(SALIDA_DIR))
else:
    SALIDA_DIR.mkdir(parents=True, exist_ok=False)
    for nombre in ['dim_municipio', 'suelo_municipio_anio', 'sin_correspondencia']:
        TABLAS[nombre].to_parquet(SALIDA_DIR / f'{nombre}.parquet', index=False)
    con.register('unificado_df', unificado)
    con.execute(f"""
        COPY (SELECT * FROM unificado_df ORDER BY fila_fuente)
        TO {literal_sql(SALIDA_DIR / 'unificado')}
        (FORMAT parquet, PARTITION_BY (anio, codigo_departamento))
    """)
    (SALIDA_DIR / 'entorno.json').write_text(json.dumps(ENTORNO, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Escrito en', rel(SALIDA_DIR))

OK    firma_igual_a_la_publicada
No se escribe: se reutiliza kit/salidas/grupo_1/c9512e696006/silver


## 13. Relectura y verificación de la publicación

Se vuelve a leer lo que quedó en disco, igual que lo leerá GOLD. Al leer `unificado/` se fijan los tipos de las columnas de partición con `hive_types`: sin eso DuckDB convertiría `codigo_departamento=05` en el número `5`.

Controles: mismas filas y misma firma que en memoria para las cuatro tablas, `codigo_departamento` sigue siendo texto de 2 dígitos y hay una partición por cada combinación año-departamento. La consulta de ejemplo filtra Tunja usando la columna de partición, de modo que DuckDB solo abre la carpeta del departamento 15.

In [14]:
LECTURA_UNIFICADO = (f"read_parquet({literal_sql((SALIDA_DIR / 'unificado').as_posix() + '/**/*.parquet')}, "
                     "hive_partitioning = true, "
                     "hive_types = {'anio': BIGINT, 'codigo_departamento': VARCHAR})")
releido = con.execute(f"SELECT {', '.join(COLUMNAS_UNIFICADO)} FROM {LECTURA_UNIFICADO}").df()

comprobar('relectura_unificado_filas', len(releido) == len(unificado), f'{len(releido):,} filas')
comprobar('relectura_unificado_firma', firma_dataframe(releido, COLUMNAS_UNIFICADO, ORDEN['unificado']) == FIRMAS['unificado'])
comprobar('relectura_codigo_departamento_texto', todos_cumplen(releido['codigo_departamento'], r'\d{2}'))
for nombre in ['dim_municipio', 'suelo_municipio_anio', 'sin_correspondencia']:
    tabla = pd.read_parquet(SALIDA_DIR / f'{nombre}.parquet')
    comprobar(f'relectura_{nombre}', firma_dataframe(tabla, list(TABLAS[nombre].columns), ORDEN[nombre]) == FIRMAS[nombre])

carpetas = sorted({archivo.parent.relative_to(SALIDA_DIR / 'unificado').as_posix()
                   for archivo in (SALIDA_DIR / 'unificado').rglob('*.parquet')})
CONTEOS['particiones'] = len(carpetas)
combinaciones = len(unificado[['anio', 'codigo_departamento']].drop_duplicates())
comprobar('particiones_anio_departamento', len(carpetas) == combinaciones, f'{len(carpetas)} carpetas')
print('Primeras particiones:', carpetas[:4])

ejemplo = con.execute(f"""
    SELECT anio, cultivo, estado_fisico,
           sum(produccion_t)      AS produccion_t,
           sum(area_cosechada_ha) AS area_cosechada_ha
    FROM {LECTURA_UNIFICADO}
    WHERE codigo_departamento = '15' AND codigo_municipio = '15001' AND apto_rendimiento
    GROUP BY ALL
    ORDER BY anio, produccion_t DESC
    LIMIT 10
""").df()
display(ejemplo)

OK    relectura_unificado_filas · 8 filas
OK    relectura_unificado_firma
OK    relectura_codigo_departamento_texto
OK    relectura_dim_municipio
OK    relectura_suelo_municipio_anio
OK    relectura_sin_correspondencia
OK    particiones_anio_departamento · 5 carpetas
Primeras particiones: ['anio=2021/codigo_departamento=05', 'anio=2021/codigo_departamento=15', 'anio=2021/codigo_departamento=25', 'anio=2021/codigo_departamento=54']


,anio,cultivo,estado_fisico,produccion_t,area_cosechada_ha
0,2021,Papa,En fresco,1100.0,40.0
1,2021,Arveja,En fresco,25.0,5.0


## 14. Resultado esperado del fixture

El fixture (`soluciones/plan_grupo_1/fixtures/raw/`) trae `esperado_silver.json` con los resultados calculados a mano (ver `fixtures/README.md`). Si existe, se comparan los conteos y algunos valores puntuales **leídos de la publicación**, no de memoria. Con datos reales el archivo no existe y la sección se omite.

In [15]:
def iguales(obtenido, esperado):
    if esperado is None or pd.isna(esperado):
        return obtenido is None or pd.isna(obtenido)
    if isinstance(esperado, float):
        return bool(np.isclose(obtenido, esperado, rtol=0, atol=1e-9))
    return obtenido == esperado


ruta_esperado = ENTRADA_DIR / 'esperado_silver.json'
if not ruta_esperado.is_file():
    print('Sin esperado_silver.json: ejecución con datos reales, se omite esta comparación.')
else:
    esperado = json.loads(ruta_esperado.read_text(encoding='utf-8'))
    suelo_publicado = pd.read_parquet(SALIDA_DIR / 'suelo_municipio_anio.parquet')
    obtenido = {**CONTEOS, 'filas_suelo': len(suelo_publicado),
                'filas_sin_correspondencia': len(pd.read_parquet(SALIDA_DIR / 'sin_correspondencia.parquet'))}
    for clave, valor in esperado.items():
        if clave in obtenido:
            comprobar(f'esperado_{clave}', iguales(obtenido[clave], valor), f'{obtenido[clave]} (esperado {valor})')
    for fila in esperado['suelo']:
        real = suelo_publicado.set_index(['codigo_municipio', 'anio']).loc[(fila['codigo_municipio'], fila['anio'])]
        comprobar(f"esperado_suelo_{fila['codigo_municipio']}_{fila['anio']}",
                  iguales(int(real['muestras']), fila['muestras']) and iguales(real['mediana_ph'], fila['mediana_ph']),
                  f"muestras {int(real['muestras'])}, mediana_ph {real['mediana_ph']}")
    m2 = releido.loc[releido['fila_fuente'].eq(2), 'area_cosechada_m2'].iloc[0]
    comprobar('esperado_area_m2_fila_2', iguales(float(m2), esperado['area_cosechada_m2_fila_2']), f'{m2:,.0f} m²')

OK    esperado_filas_eva_entrada · 8 (esperado 8)
OK    esperado_filas_unificado · 8 (esperado 8)
OK    esperado_filas_aptas_rendimiento · 7 (esperado 7)
OK    esperado_eva_sin_divipola · 1 (esperado 1)
OK    esperado_agrosavia_enlazadas · 6 (esperado 6)
OK    esperado_agrosavia_sin_enlace · 1 (esperado 1)
OK    esperado_agrosavia_por_alias · 2 (esperado 2)
OK    esperado_filas_suelo · 5 (esperado 5)
OK    esperado_filas_sin_correspondencia · 2 (esperado 2)
OK    esperado_particiones · 5 (esperado 5)
OK    esperado_cobertura_suelo_pct · 75.0 (esperado 75.0)
OK    esperado_produccion_total_t · 1809.0 (esperado 1809.0)
OK    esperado_area_cosechada_total_ha · 165.0 (esperado 165.0)
OK    esperado_suelo_15001_2021 · muestras 2, mediana_ph 5.5
OK    esperado_suelo_15001_2022 · muestras 1, mediana_ph nan
OK    esperado_suelo_54001_2021 · muestras 1, mediana_ph 7.0
OK    esperado_area_m2_fila_2 · 100,000 m²


## 15. Limpieza de memoria y caché

Paso del diagrama: *se limpia de memoria la información de CSV y Parquet para optimizar futuros procesamientos*. Las tablas grandes ya están publicadas y verificadas, así que se liberan:

1. `con.close()` cierra DuckDB y libera su memoria intermedia;
2. `del` elimina los DataFrames grandes y `gc.collect()` recoge los objetos sin referencia;
3. `malloc_trim(0)` pide al asignador de glibc (Linux/WSL) que devuelva al sistema la memoria libre; en otros sistemas se omite.

Se mide `RssAnon` antes y después. Es un dato **informativo**: el asignador puede conservar parte de la memoria y eso no invalida la publicación. Después de esta celda ya no se usan `eva`, `agrosavia`, `unificado` ni `releido`.

In [16]:
MEMORIA_ANTES = memoria_mb()
con.close()
del eva, divipola, agrosavia, agro, agro_enlazada, eva_dim, unificado, releido, tabla, TABLAS
gc.collect()
try:
    ctypes.CDLL('libc.so.6').malloc_trim(0)
except (OSError, AttributeError):
    pass
MEMORIA_DESPUES = memoria_mb()

if MEMORIA_ANTES is None:
    print('Medición de memoria no disponible en este sistema (requiere /proc).')
else:
    print(f'Memoria de trabajo (RssAnon): {MEMORIA_ANTES:,.1f} MB → {MEMORIA_DESPUES:,.1f} MB '
          f'(liberados {MEMORIA_ANTES - MEMORIA_DESPUES:,.1f} MB)')

Memoria de trabajo (RssAnon): 82.3 MB → 79.3 MB (liberados 3.0 MB)


## 16. Controles y registro de la ejecución

`controles_silver.json` reúne identificador, firmas, conteos, memoria, resultado de cada control y el bloque `ENTORNO`. Su `estado` es `valida` solo si **todos** los controles pasaron; como `comprobar` detiene el notebook ante cualquier fallo, llegar hasta aquí ya lo garantiza.

Para una ejecución nueva se escribe `controles_silver.json` en la carpeta y se agrega **una línea** a `registro_silver.jsonl` con lo necesario para reconocerla después (firma de entrada y de contenido, versiones, commit y responsable). Escribir los controles al final marca la carpeta como publicación válida: una carpeta sin `controles_silver.json` es un intento fallido y GOLD no debe usarla. Si la entrada ya estaba procesada, no se modifica nada.

In [17]:
CONTROLES_SILVER = {
    'run_id': RUN_ID,
    'fecha': datetime.now(ZONA).isoformat(timespec='seconds'),
    'estado': 'valida' if all(CONTROLES.values()) else 'fallida',
    'version_notebook': VERSION_NOTEBOOK,
    'version_reglas': VERSION_REGLAS,
    'entrada': rel(ENTRADA_DIR),
    'salida': rel(SALIDA_DIR),
    'firma_entrada': FIRMA_ENTRADA,
    'firma_contenido': FIRMA_CONTENIDO,
    'firmas': FIRMAS,
    **CONTEOS,
    'memoria_mb_antes': MEMORIA_ANTES,
    'memoria_mb_despues': MEMORIA_DESPUES,
    'controles': CONTROLES,
    'errores': [],
    'entorno': ENTORNO,
}

if YA_PROCESADO:
    print('Ejecución reutilizada: controles_silver.json y registro existentes no se modifican.')
else:
    (SALIDA_DIR / 'controles_silver.json').write_text(
        json.dumps(CONTROLES_SILVER, ensure_ascii=False, indent=2), encoding='utf-8')
    linea = {
        'run_id': RUN_ID,
        'fecha': CONTROLES_SILVER['fecha'],
        'firma_entrada': FIRMA_ENTRADA,
        'firma_contenido': FIRMA_CONTENIDO,
        'estado': CONTROLES_SILVER['estado'],
        'ruta_salida': rel(SALIDA_DIR),
        'version_notebook': VERSION_NOTEBOOK,
        'version_reglas': VERSION_REGLAS,
        'python': ENTORNO['python']['version'],
        'commit': ENTORNO['git']['commit'],
        'responsable': RESPONSABLE,
    }
    with REGISTRO.open('a', encoding='utf-8') as archivo:
        archivo.write(json.dumps(linea, ensure_ascii=False) + '\n')
    print(f"Publicación SILVER {CONTROLES_SILVER['estado']}: {rel(SALIDA_DIR)}")
print(f"{sum(CONTROLES.values())} de {len(CONTROLES)} controles superados.")

Ejecución reutilizada: controles_silver.json y registro existentes no se modifican.
49 de 49 controles superados.


## 17. Resumen y entrega a GOLD

Tabla final para la sustentación y la ruta que Julián debe poner en su `ENTRADA_DIR`. Recuerda los límites de esta capa:

- el suelo se asocia por **municipio-año**, no por parcela: es contexto, no explicación del rendimiento;
- la cobertura de suelo depende del solape de años entre AGROSAVIA (2014–2025) y EVA (2019–2025) y de que el municipio tenga muestras;
- los alias son una regla escrita y versionada; cualquier cambio debe subir `VERSION_REGLAS`.

In [18]:
resumen = pd.Series({
    'Filas EVA (entrada = unificado)': f"{CONTEOS['filas_unificado']:,}",
    'Filas aptas para rendimiento': f"{CONTEOS['filas_aptas_rendimiento']:,}",
    'Códigos EVA sin DIVIPOLA': f"{CONTEOS['eva_sin_divipola']:,}",
    'Muestras AGROSAVIA enlazadas': (f"{CONTEOS['agrosavia_enlazadas']:,} de {CONTEOS['agrosavia_entrada']:,} "
                                     f"({CONTEOS['agrosavia_por_alias']:,} por alias)"),
    'Filas EVA con datos de suelo': f"{CONTEOS['cobertura_suelo_pct']:.2f} %",
    'Producción total (t)': f"{CONTEOS['produccion_total_t']:,.1f}",
    'Área cosechada total (ha)': f"{CONTEOS['area_cosechada_total_ha']:,.1f}",
    'Particiones anio/departamento': CONTEOS['particiones'],
    'Estado': CONTROLES_SILVER['estado'],
    'Run': RUN_ID,
}, name='valor').to_frame()
display(resumen)
print('Para GOLD (Julián):')
print(f"ENTRADA_DIR = REPOSITORIO / '{rel(SALIDA_DIR)}'")

,valor
Filas EVA (entrada = unificado),8
Filas aptas para rendimiento,7
Códigos EVA sin DIVIPOLA,1
Muestras AGROSAVIA enlazadas,6 de 7 (2 por alias)
Filas EVA con datos de suelo,75.00 %
Producción total (t),"1,809.0"
Área cosechada total (ha),165.0
Particiones anio/departamento,5
Estado,valida
Run,c9512e696006


Para GOLD (Julián):
ENTRADA_DIR = REPOSITORIO / 'kit/salidas/grupo_1/c9512e696006/silver'
